# BeverageDzAI — GPU embedding generation

This notebook generates BGE-M3 dense vectors for the preprocessed public patent corpus. Upload only `chunks-20k.jsonl`. **Never upload `.env` or the Google service-account JSON.** The downloaded ZIP is imported into the local Qdrant instance afterward.

## 1. Enable a GPU
In Colab select **Runtime → Change runtime type → T4 GPU** (or a better available GPU), then run the cells in order.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU detected. Enable a GPU runtime before continuing.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
!pip -q install 'sentence-transformers>=3.0,<6' 'numpy>=1.26,<3'

## 2. Upload the chunk file
Upload `innovation-rag/data/processed/chunks/chunks-20k.jsonl`. It contains public patent text only and no credentials.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one chunks-20k.jsonl file.'
CHUNKS_PATH = next(iter(uploaded))
assert CHUNKS_PATH.endswith('.jsonl')
print('Uploaded:', CHUNKS_PATH)

In [ ]:
import hashlib
import json
from pathlib import Path

chunk_ids = []
texts = []
with Path(CHUNKS_PATH).open('r', encoding='utf-8') as handle:
    for line_number, line in enumerate(handle, start=1):
        if not line.strip():
            continue
        record = json.loads(line)
        chunk_ids.append(record['id'])
        texts.append(record['text'])

assert chunk_ids and len(chunk_ids) == len(set(chunk_ids)), 'Chunk IDs must be non-empty and unique.'
ordered_ids_hash = hashlib.sha256('\n'.join(chunk_ids).encode('utf-8')).hexdigest()
print(f'Loaded {len(texts):,} chunks')
print('Ordered IDs SHA-256:', ordered_ids_hash)

## 3. Generate normalized BGE-M3 vectors
The output order exactly matches the JSONL line order. Mixed precision is used on CUDA to reduce VRAM and improve throughput.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'BAAI/bge-m3'
BATCH_SIZE = 16  # Reduce to 8 if Colab reports an out-of-memory error.
model = SentenceTransformer(MODEL_NAME, device='cuda')
model.half()
dense_embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype(np.float32, copy=False)
assert dense_embeddings.shape[0] == len(chunk_ids)
print('Embedding matrix:', dense_embeddings.shape, dense_embeddings.dtype)

## 4. Package and download
The manifest lets the local importer reject vectors generated from a different model, chunk count, or chunk ordering.

In [ ]:
from datetime import datetime, timezone
import zipfile

EMBEDDINGS_FILE = 'dense_embeddings_bge_m3.npy'
MANIFEST_FILE = 'dense_embeddings_manifest.json'
ARCHIVE_FILE = 'beverage_dense_embeddings_bge_m3.zip'
np.save(EMBEDDINGS_FILE, dense_embeddings, allow_pickle=False)
manifest = {
    'format_version': 1,
    'model': MODEL_NAME,
    'chunk_count': len(chunk_ids),
    'dimensions': int(dense_embeddings.shape[1]),
    'normalized': True,
    'ordered_chunk_ids_sha256': ordered_ids_hash,
    'created_at': datetime.now(timezone.utc).isoformat(),
}
Path(MANIFEST_FILE).write_text(json.dumps(manifest, indent=2), encoding='utf-8')
with zipfile.ZipFile(ARCHIVE_FILE, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(EMBEDDINGS_FILE)
    archive.write(MANIFEST_FILE)
print(json.dumps(manifest, indent=2))
print('Archive size (MB):', round(Path(ARCHIVE_FILE).stat().st_size / 1024**2, 1))
files.download(ARCHIVE_FILE)